## Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import re

## Importing data

Columns to drop

In [ ]:
columns_to_not_use = [
# Same values for all rows
    'record_type' ,
    'record_subclass' ,
    'confirmation_location' ,
    'dia_bp' ,
    'group_number' ,
    'manufacturer_code' ,
    'patient_age_units' ,
    'patient_height' ,
    'patient_weight' ,
    'reason_for_procedure' ,
    'sys_bp' ,
    'text_size' ,
    'pp_interval' ,

# No value at all
    'assisting_provider' ,
    'attending_phy' ,
    'patient_middle_name' ,
    'patient_secondary_id' ,
    'patient_ssn' ,
    'procedural_diagnosis' ,
    'referral_source' ,
    'referring_phy' ,
]

Columns to format in a particular type

In [ ]:
sure_string_cols = [
    'pseudopaz'
]
dtype = {col: 'string' for col in sure_string_cols}

string_cat_cols = [
    'site_number', # ???
    'record_status',
    'medications',
    'patient_height_units' ,
    'patient_location' ,
    'patient_race' ,
    'patient_race_str' ,
    'patient_room' ,
    'patient_sex' ,
    'patient_weight_units' ,
    'transmit_device_number' ,
    'reason_for_change' ,
]
dtype.update({col: 'string' for col in string_cat_cols})

Import the excel file

In [ ]:
df = pd.read_excel('../raw_data/dati_escribe.xlsx',
    usecols=lambda col: col not in columns_to_not_use,
    dtype_backend='numpy_nullable', # For allowing the `string` dtype
    dtype=dtype
)

# Make the index start from 2 (as the original excel)
df.index = df.index + 2

## Parsing and reformatting

Defining `format_phy_name` function

In [ ]:
def format_phy_name(value):
    """
    Requires pandas.
    TODO
    """
    # Handle NA values
    if pd.isna(value):
        return pd.NA

    # Count whitespaces
    dot_count = value.count('.')

    # Handle the only name with one dot
    if dot_count == 1:
        return value

    elif dot_count == 2:
        second_dot = value.rfind('.')
        name = value[second_dot -1 : second_dot]
        surname = value[value.rfind('.') +1 : ].strip()
        formatted_name = 'DR-' + name + '_' + surname
        
        return formatted_name

### Formatting `overreading_phy` column

In [ ]:
# Only name with one dot
df['overreading_phy'] = df['overreading_phy'].replace('DR. GP BROSO', 'DR-GP_BROSO')

# Run the function
df['overreading_phy'] = df['overreading_phy'].apply(format_phy_name).astype('string')

### Splitting and formatting `interp_text` column

In [ ]:
def split_column(value):

    # If row is already a NA value, leave unchanged
    if pd.isna(value):
        return pd.Series([pd.NA, pd.NA, pd.NA])

    # Split on the first ~~ from right
    parts = value.rsplit('~~', maxsplit=1)

    # Place a NA in the missing splitted parts
    for i, part in enumerate(parts):
        if part == '':
            parts[i] = pd.NA

    # Parse "parts[1]"
    if parts[1].startswith('REPORT NON CONFERM'):
        doc, datetime = 'RNC', pd.NaT
    elif parts[1].startswith('Refertato da _________________'):
        doc, datetime = 'phy_NA', pd.NaT
    elif parts[1].startswith('Refertato da '):
        doc_pro, date, hour = (re.sub(' +', ' ', parts[1])).strip().rsplit(' ', maxsplit=2)
        doc = format_phy_name(doc_pro[13:])
        datetime = pd.to_datetime('{} {}'.format(date, hour), format='%d/%m/%Y %H.%M.%S')

    if len(parts) == 2:
        return pd.Series([parts[0], doc, datetime])
    else:
        raise Exception('Abnormal row')

Pre-format row 1685, the only problematic

In [ ]:
original = df.at[1685, 'interp_text']
df.at[1685, 'interp_text'] = '~' + original

In [ ]:
# Get position and split
original_col_name = 'interp_text'
original_position = df.columns.get_loc(original_col_name)

# Split the column
split_df = df[original_col_name].apply(split_column)
split_df.columns = ['diagnosis', 'phy', 'phy_date']
split_df[['diagnosis', 'phy']] = split_df[['diagnosis', 'phy']].astype('string')

# Reconstruct dataframe: columns before + new columns + columns after
df = pd.concat([
    df.iloc[:, :original_position
        # +1 # Decomment for keeping the original column
    ],
    split_df,
    df.iloc[:, original_position+1:]
], axis=1)

### Merging similar values in columns

#### `comment_field`

Values are `{..., 'Dolore toracico': 8,'Dolore torace': 2, ...}` - `Dolore torace` values are merged to `Dolore toracico`

In [ ]:
df['comment_field'] = df['comment_field'].replace('Dolore torace', 'Dolore toracico')

#### `note1`

Merging:
- `DOL. TORACICO` and `DOL TOR` and `DOLORETORAC` and `DOLORE TORACICO`
    - to `TORACALGIA`
- `ADDOMINALG` and `DOLORE ADDOMIN`
    - to `ADDOMINALGIA`
- `Dispnea`
    - to `DISPNEA`

In [ ]:
df['note1'] = df['note1'].replace(['DOLORE TORACICO', 'DOLORETORAC', 'DOL. TORACICO', 'DOL TOR'], 'TORACALGIA')
df['note1'] = df['note1'].replace(['ADDOMINALG', 'DOLORE ADDOMIN'], 'ADDOMINALGIA')
df['note1'] = df['note1'].replace(['Dispnea'], 'DISPNEA')

#### `note2` (tanti auguri)

Merging:
- `.` and `.`
    - to `.`

In [ ]:
# df[''] = df[''].replace([''], '')

#### `patient_location`

Merging:
- `OTO`
    - to `Oto`

In [ ]:
df['patient_location'] = df['patient_location'].replace(['OTO'], 'Oto')

#### ``

Merging:
- `.` and `.`
    - to `.`

In [ ]:
# df[''] = df[''].replace([''], '')

### Formatting `patient_date_of_birth` column

Since 33 rows have the date formatted as `06/17/1950` (for example row 790), `format='mixed'` is required

In [ ]:
df['patient_date_of_birth'] = pd.to_datetime(df['patient_date_of_birth'], errors='raise', dayfirst=True, format='mixed')

## Final checks

In [ ]:
df

In [ ]:
df.info()

## Export

In [ ]:
df.to_parquet("../raw_data/parquets/dati_escribe.parquet"
    # , compression="gzip"
)

### Inconsistency between `pseudopaz` and other columns

#### Proof of the inconsistency

`patient_age`, `patient_date_of_birth`, `patient_race`, `patient_sex` are inconsistent

In [ ]:
# (df.groupby('pseudopaz')['patient_age'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_date_of_birth'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_race'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_sex'].nunique() == 1).all()

#### See which `pseudopaz` have multiple characteristics that should be unique

In [ ]:
# incons = df.groupby('pseudopaz')[''].nunique()
# incons = incons[incons > 1].sort_values(ascending=False)
# incons